# E1.2 · Building the AI and agent inventory

**Function E — AI Governance for Agentic Systems → Building the Governance Framework — Risk and Control**  ·  *Security of AI*

Builds on **[E1.1 · Why point-in-time control testing fails for AI](https://spbreed.github.io/cyber-commons/lessons/E1.1.html)**.

| | |
|---|---|
| Open-source tooling | agentgateway, SPIRE |
| Open-weight models | — |
| Frontier models | — |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off — and where a lesson involves a model, the same code calls an open-weight endpoint or a frontier API when you configure one.

## 1 · The hook

Nobody can govern what nobody has listed. The inventory is the least interesting artefact in this function and the one everything else depends on — and the hard part is not building it, it is keeping it true next quarter.

> **At CyberTravels.** Nobody at CyberTravels can currently list every agent, MCP server and vector index in the estate. Everything else in this chapter depends on that list being true next quarter.

## 2 · The framework

```
   what has to be in the register

   model . agent . integration . MCP server . eval corpus
        |
   +----v-------------------------------------------+
   | owner . purpose . data classes . autonomy level|
   | tools it holds . environment . last verified   |
   +------------------------------------------------+

   building it is a project. keeping it true is the control.
```

You cannot govern, tier, test or revoke what you cannot list. The AI inventory is
therefore the first control, not a documentation exercise.

The honest finding of every first inventory is the same: **most of it was already
in production.** Not because anyone was reckless, but because AI features arrive
inside products you already bought, and agents get created programmatically by
other agents.

Three sources, and the third finds what the first two miss:

1. the **model registry** — what your ML team registered,
2. **procurement and expense** — what someone bought,
3. **egress logs to model-provider domains** — what is actually being used.

Source 3 is the one that discovers the department using a frontier API on a
personal card, and the SaaS product that quietly added an AI feature.

## 3 · Demo — build the inventory from three sources

In [ ]:
from dataclasses import dataclass, field

@dataclass
class AIAsset:
    name: str
    kind: str              # model | agent | copilot | embedded-feature
    owner: str = ""
    autonomy: str = "L1"
    data: tuple = ()
    external: bool = False
    discovered_via: str = "registry"

    def gaps(self):
        g = []
        if not self.owner:
            g.append("no named owner — nobody can accept the risk or recertify it")
        if self.discovered_via != "registry":
            g.append(f"not registered — found via {self.discovered_via}")
        if self.autonomy in ("L2.5", "L3") and not self.owner:
            g.append("acts semi-autonomously with nobody accountable")
        return g

REGISTRY = [
 AIAsset("fraud-scoring-model", "model", "risk-eng", "L1", ("customer",)),
 AIAsset("support-summariser", "copilot", "support-eng", "L1", ("customer",)),
]
PROCUREMENT = [
 AIAsset("vendor-contract-analyser", "embedded-feature", "", "L1", ("regulated",),
         discovered_via="expense report"),
]
EGRESS = [
 AIAsset("unknown-openai-usage-marketing", "copilot", "", "L1", ("public",),
         discovered_via="egress logs"),
 AIAsset("pr-remediation-agent", "agent", "", "L2.5", ("customer",), True,
         discovered_via="egress logs"),
 AIAsset("agent-worker-7f3c", "agent", "", "L2.5", ("customer",),
         discovered_via="egress logs"),
]
ALL = REGISTRY + PROCUREMENT + EGRESS
print(f"{'asset':34s}{'kind':18s}{'autonomy':10s}{'owner':14s}found via")
print("-" * 92)
for a in ALL:
    print(f"{a.name:34s}{a.kind:18s}{a.autonomy:10s}{a.owner or '—':14s}{a.discovered_via}")
print(f"\nregistry found {len(REGISTRY)}; the other two sources found "
      f"{len(ALL)-len(REGISTRY)} more.")

## 4 · Where it breaks — the gap distribution is always like this

In [ ]:
unowned = [a for a in ALL if not a.owner]
unregistered = [a for a in ALL if a.discovered_via != "registry"]
high_autonomy_unowned = [a for a in ALL if a.autonomy in ("L2.5","L3") and not a.owner]

print(f"assets                    {len(ALL)}")
print(f"no named owner            {len(unowned)}  {[a.name for a in unowned]}")
print(f"never registered          {len(unregistered)}")
print(f"L2.5+ with no owner       {len(high_autonomy_unowned)}  "
      f"{[a.name for a in high_autonomy_unowned]}")

print("\ngaps in detail:")
for a in ALL:
    for g in a.gaps():
        print(f"   {a.name:34s}{g}")
assert high_autonomy_unowned

## 5 · The control — a discovery query you can re-run

In [ ]:
MODEL_PROVIDER_DOMAINS = {"api.openai.com", "api.anthropic.com",
                          "generativelanguage.googleapis.com",
                          "api.mistral.ai", "api.together.xyz"}

EGRESS_LOG = [
 {"src": "marketing-workstation-14", "host": "api.openai.com", "bytes": 240_000},
 {"src": "svc-pr-remediation",       "host": "api.anthropic.com", "bytes": 8_400_000},
 {"src": "build-runner-3",           "host": "registry.npmjs.org", "bytes": 90_000},
 {"src": "agent-worker-7f3c",        "host": "api.together.xyz", "bytes": 1_200_000},
]
def discover(log, known_names):
    found = []
    for row in log:
        if row["host"] not in MODEL_PROVIDER_DOMAINS: continue
        if row["src"] in known_names: continue
        found.append({"source": row["src"], "provider": row["host"],
                      "volume": row["bytes"],
                      "finding": "AI usage not present in the inventory"})
    return found

known = {a.name for a in REGISTRY + PROCUREMENT}
for f in discover(EGRESS_LOG, known):
    print(f"{f['source']:30s}{f['provider']:34s}{f['volume']:>10,} bytes")
    print(f"{'':30s}{f['finding']}")

def inventory_health(assets):
    return {"total": len(assets),
            "owned": sum(1 for a in assets if a.owner),
            "registered": sum(1 for a in assets if a.discovered_via == "registry"),
            "coverage": round(sum(1 for a in assets if a.owner)/len(assets), 2)}
print(f"\n{inventory_health(ALL)}")

## What you just proved

The registry lists 2 assets; procurement and egress logs find 4 more. Four assets have no owner, four were never registered, and two L2.5-autonomy agents have nobody accountable. The egress query identifies three sources talking to model providers that are absent from the inventory, giving an ownership coverage of 0.33.

## Your turn

Run the egress query for real: one week of traffic to model-provider domains, joined against your inventory. It takes an hour and it always finds something.

---

**Next → [E1.3 · Risk tiering agentic use cases](https://spbreed.github.io/cyber-commons/lessons/E1.3.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/E1.2.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/E1.2.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*